# Limpieza y Deduplicación -> Capa Silver
Aplicación de reglas de limpieza y guardado en formato Delta.

In [ ]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window

base_path = 'file:/Workspace/Repos/your_email/castor-data-engineer/data'

df_user = spark.read.parquet(f'{base_path}/bronze/usuarios')
df_trans = spark.read.parquet(f'{base_path}/bronze/transacciones')

In [ ]:
# Limpieza usuarios
# Deduplicar por id_usuario tomando el más reciente (si hubiera un timestamp, aquí se usa row_number())
df_user_clean = df_user.dropDuplicates(['id_usuario'])

# Imputación de nulos según README
df_user_clean = df_user_clean.fillna({'pais': 'Desconocido'})
mediana_edad = df_user_clean.approxQuantile('edad', [0.5], 0.01)[0]
df_user_clean = df_user_clean.fillna({'edad': mediana_edad})

In [ ]:
# Limpieza transacciones
df_trans_clean = df_trans.withColumn('monto_clean', F.regexp_replace(F.col('monto'), r'[\$,\s]', '').cast('double')) \
                         .withColumn('fecha_transaccion', F.to_date(F.from_unixtime('timestamp_unix')))

df_trans_clean = df_trans_clean.fillna({'categoria': 'Sin categoría'})

In [ ]:
# Guardar en Silver como Delta (Si no tienes Delta instalado localmente, puedes cambiar a parquet para probar)
try:
    df_user_clean.write.format('delta').mode('overwrite').save(f'{base_path}/silver/usuarios')
    df_trans_clean.write.format('delta').mode('overwrite').save(f'{base_path}/silver/transacciones')
except Exception:
    print('Delta no disponible, guardando como parquet')
    df_user_clean.write.mode('overwrite').parquet(f'{base_path}/silver/usuarios')
    df_trans_clean.write.mode('overwrite').parquet(f'{base_path}/silver/transacciones')